In [18]:
import sys
import os
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm import tqdm
import os
import random
import numpy as np
import tensorflow as tf

# Set the project root (one level up from /notebooks/)
project_root = Path(os.getcwd()).parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import your classes and functions directly from the source files
from src.dataset.eyes_dataset import build_dataloaders, EyesDataset
from src.models.binary_roi_classifier import build_binary_classifier

In [19]:
SEED = 42

# Python random
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# TensorFlow
tf.random.set_seed(SEED)

In [20]:
STAGE_1_EPOCHS     = 50
STAGE_2_EPOCHS     = 50
BATCH_SIZE         = 64
STAGE_1_LR         = 1e-3
STAGE_2_HEAD_LR    = 1e-3
STAGE_2_BACKBONE_LR = 1e-4
MIXUP_ALPHA        = 0.4
EARLY_STOP_PATIENCE = 50
OUTPUT_BEST_MODEL_NAME = "best_model_oversampled_1.pt"
 
DATASET_ROOTS = {
    "eyes":  "../dataset_split/train/eyes",
    "mouth": "../dataset_split/train/mouth",
}
 
CLASS_LABELS = {
    "eyes":  {"closed": 0, "open": 1},
    "mouth": {"closed": 0, "open": 1},
}
 
CHECKPOINT_DIR = Path("runs/binary")

In [15]:
from pathlib import Path

roi = "eyes"
root_path = Path(DATASET_ROOTS[roi])
print(f"Checking root: {root_path.absolute()}")

for class_name in CLASS_LABELS[roi].keys():
    class_dir = root_path / class_name
    images = list(class_dir.glob("*.png"))
    print(f"Folder '{class_name}': Found {len(images)} .png images")

Checking root: c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset_split\train\eyes
Folder 'closed': Found 2296 .png images
Folder 'open': Found 21042 .png images


In [16]:
# =========================
# Sanity check
# =========================
def check_dataset(roi):
    root_path = Path(DATASET_ROOTS[roi])
    print(f"Checking root: {root_path.absolute()}")
    for class_name in CLASS_LABELS[roi].keys():
        class_dir = root_path / class_name
        images = list(class_dir.glob("*.png"))
        print(f"  Folder '{class_name}': Found {len(images)} .png images")

In [21]:
def mixup_batch(images, labels, alpha):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size = images.size(0)
    perm = torch.randperm(batch_size, device=images.device)
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam

def mixup_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss  = total_loss / len(loader.dataset)
    acc       = accuracy_score(all_labels, all_preds)
    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, precision, recall, f1

In [22]:
def run_stage(
    model, optimizer, criterion,
    train_loader, val_loader,
    device, epochs, start_epoch,
    use_mixup, checkpoint_path,
):
    best_val_f1 = 0.0
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(start_epoch, start_epoch + epochs):
        model.train()
        running_loss   = 0.0
        correct        = 0
        total          = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup:
                mixed, la, lb, lam = mixup_batch(images, labels, MIXUP_ALPHA)
                logits = model(mixed)
                loss = mixup_loss(criterion, logits, la, lb, lam)
            else:
                logits = model(images)
                loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct      += (logits.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc  = correct / total

        val_loss, val_acc, prec, rec, f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(f1)

        print(
            f"Epoch {epoch:>3} | "
            f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f} | "
            f"prec={prec:.3f}  rec={rec:.3f}  F1={f1:.3f}"
        )

        if f1 > best_val_f1:
            best_val_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> Saved best model (F1={f1:.3f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print("Early stop triggered.")
                break

    return history

In [23]:
def plot_history(history: dict, roi: str, save_dir: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Training History — {roi.upper()} (Over-sampled)", fontsize=13)

    # --- Loss ---
    axes[0].plot(epochs, history["train_loss"], label="Train Loss", color="steelblue")
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   color="tomato")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # --- Accuracy ---
    axes[1].plot(epochs, history["train_acc"], label="Train Acc", color="steelblue")
    axes[1].plot(epochs, history["val_acc"],   label="Val Acc",   color="tomato")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = save_dir / "oversampled_training_history.png"
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved training history -> {save_path}")

In [24]:
def build_sampler(all_samples, train_indices, num_classes):
    targets = torch.tensor([all_samples[i][1] for i in train_indices])
    class_counts  = torch.bincount(targets, minlength=num_classes).float()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]

    print(f"  Class counts  : {class_counts.tolist()}")
    print(f"  Class weights : {[f'{w:.4f}' for w in class_weights.tolist()]}")

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    total = len(sample_weights)
    for cls in range(num_classes):
        cls_weight = class_weights[cls].item()
        effective = int(round(cls_weight / class_weights.sum().item() * total))
        print(f"  Class {cls} effective samples : ~{effective} / {total}")
    return sampler, class_weights

In [26]:
def train_roi(roi):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*50}")
    print(f"Training ROI: {roi.upper()} on {device}")
    print(f"{'='*50}")

    check_dataset(roi)

    root        = Path(DATASET_ROOTS[roi])
    num_classes = len(CLASS_LABELS[roi])

    # --- 1. Get all samples for sampler construction ---
    all_samples = EyesDataset(root, CLASS_LABELS[roi], transform=None).samples

    # --- 2. Stratified split, no sampler yet — just to get train_indices ---
    _, val_loader, train_indices = build_dataloaders(
        root=root,
        class_to_label=CLASS_LABELS[roi],
        batch_size=BATCH_SIZE,
        sampler=None,
    )

    # --- 3. Build sampler + class weights from train split only ---
    sampler, class_weights = build_sampler(all_samples, train_indices, num_classes)

    # --- 4. Rebuild train loader with sampler ---
    train_loader, _, _ = build_dataloaders(
        root=root,
        class_to_label=CLASS_LABELS[roi],
        batch_size=BATCH_SIZE,
        sampler=sampler,
    )

    # --- 5. Weighted loss ---
    criterion = nn.CrossEntropyLoss()

    # --- 6. Model ---
    model = build_binary_classifier(pretrained=True, freeze_backbone=True).to(device)
    checkpoint_dir  = CHECKPOINT_DIR / roi
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / OUTPUT_BEST_MODEL_NAME

    # --- Stage 1: Frozen backbone ---
    print("\n--- Stage 1: Frozen Backbone ---")
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=STAGE_1_LR,
    )
    history_s1 = run_stage(
        model, optimizer, criterion,
        train_loader, val_loader, device,
        epochs=STAGE_1_EPOCHS, start_epoch=1,
        use_mixup=False,
        checkpoint_path=checkpoint_path,
    )

    # --- Stage 2: Fine-tune ---
    print("\n--- Stage 2: Fine-tuning ---")
    model.freeze_backbone(toggle=False, last_n_blocks=2)
    optimizer = torch.optim.Adam([
        {"params": model.backbone.parameters(), "lr": STAGE_2_BACKBONE_LR},
        {"params": model.head.parameters(),     "lr": STAGE_2_HEAD_LR},
    ])
    history_s2 = run_stage(
        model, optimizer, criterion,
        train_loader, val_loader, device,
        epochs=STAGE_2_EPOCHS, start_epoch=STAGE_1_EPOCHS + 1,
        use_mixup=True,
        checkpoint_path=checkpoint_path,
    )

    print(f"\nDone. Best model saved to: {checkpoint_path}")

    full_history = {
        key: history_s1[key] + history_s2[key]
        for key in history_s1
    }
    results_dir = Path("runs/eval") / roi
    results_dir.mkdir(parents=True, exist_ok=True)
    plot_history(full_history, roi, results_dir)

In [27]:
train_roi("eyes")


Training ROI: EYES on cuda
Checking root: c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset_split\train\eyes
  Folder 'closed': Found 2296 .png images
  Folder 'open': Found 21042 .png images
  Class counts  : [1952.0, 17886.0]
  Class weights : ['0.0005', '0.0001']
  Class 0 effective samples : ~17886 / 19838
  Class 1 effective samples : ~1952 / 19838

--- Stage 1: Frozen Backbone ---


Epoch   1 | train_loss=0.1772  train_acc=0.933 | val_loss=0.1710  val_acc=0.939 | prec=0.817  rec=0.869  F1=0.840
  -> Saved best model (F1=0.840)


Epoch   2 | train_loss=0.1347  train_acc=0.949 | val_loss=0.1334  val_acc=0.948 | prec=0.869  rec=0.825  F1=0.845
  -> Saved best model (F1=0.845)


Epoch   3 | train_loss=0.1164  train_acc=0.954 | val_loss=0.1396  val_acc=0.944 | prec=0.824  rec=0.918  F1=0.863
  -> Saved best model (F1=0.863)


Epoch   4 | train_loss=0.1049  train_acc=0.959 | val_loss=0.1455  val_acc=0.947 | prec=0.831  rec=0.927  F1=0.871
  -> Saved best model (F1=0.871)


...

Epoch  96 | train_loss=0.2578  train_acc=0.773 | val_loss=0.0941  val_acc=0.977 | prec=0.919  rec=0.957  F1=0.937


Epoch  97 | train_loss=0.2633  train_acc=0.749 | val_loss=0.0954  val_acc=0.975 | prec=0.913  rec=0.952  F1=0.932


Epoch  98 | train_loss=0.2435  train_acc=0.758 | val_loss=0.1015  val_acc=0.976 | prec=0.919  rec=0.954  F1=0.936


Epoch  99 | train_loss=0.2366  train_acc=0.750 | val_loss=0.1049  val_acc=0.975 | prec=0.912  rec=0.959  F1=0.934


Epoch 100 | train_loss=0.2602  train_acc=0.762 | val_loss=0.0918  val_acc=0.976 | prec=0.915  rec=0.959  F1=0.936

Done. Best model saved to: runs\binary\eyes\best_model_oversampled_1.pt
  Saved training history -> runs\eval\eyes\oversampled_training_history.png


In [ ]:
from torchview import draw_graph

model = build_binary_classifier(
    pretrained=False,
    freeze_backbone=True
)

# Only visualize classifier head
head = model.head
head.eval()

# Input to head = pooled backbone output
dummy_input = torch.randn(1, 576)

graph = draw_graph(
    head,
    input_data=dummy_input,
    expand_nested=True,
    save_graph=True,
    filename="cnn_head_architecture",
    directory="./diagrams"
)

graph.visual_graph.graph_attr["rankdir"] = "LR"

graph.visual_graph.render(format="png")